## 1. Cell 1 · Imports

In [2]:
## 1 · Imports

import pandas as pd
import osmnx as ox
import numpy as np

print(f"pandas  {pd.__version__}")
print(f"osmnx   {ox.__version__}")
print(f"numpy   {np.__version__}")

pandas  3.0.2
osmnx   2.1.0
numpy   2.4.4


## 2.0 Load Clean data - CSV

In [3]:
## 2 · Load Data

df = pd.read_csv("csv files/raw_umbrella_data.csv")

print(f"✓ {len(df)} rows loaded")
print(df[["osm_id", "lat", "lon", "label"]].head())

✓ 2914 rows loaded
       osm_id        lat        lon       label
0   796338420  40.758949 -73.985385      retail
1   258619240  40.759153 -73.984927      retail
2  2613887631  40.758865 -73.985491      retail
3  3595084281  40.759125 -73.985377      retail
4  1740694091  40.758775 -73.984720  food_drink


## 3.0 street map
download the entire street network for Times Square once and then match each shop to its nearest street locally.
Instead of relying on street_width_m, which is often missing in OSM data,
use highway type as a categorical feature and lanes as a numeric proxy.
The highway tag provides an almost complete street hierarchy
(primary, secondary, residential, service, etc.), while the lanes tag
gives a useful estimate of relative road width when available.
Together, highway_type and lanes provide a more reliable replacement
for street_width_m.

In [5]:
## 3 · Street Features (highway type + lanes)

# Download street network once for the whole area
print("Downloading street network...")
G = ox.graph_from_point((40.7589, -73.9851), dist=1500, network_type="drive")
print(f"✓ {len(ox.graph_to_gdfs(G, nodes=False))} street segments loaded")

# For each shop find the nearest edge and get highway type and lanes
def nearest_street_features(lat, lon):
    try:
        nearest = ox.nearest_edges(G, lon, lat)
        edge_data = G.edges[nearest]
        highway = edge_data.get("highway", None)
        lanes   = edge_data.get("lanes", None)
        if isinstance(highway, list):
            highway = highway[0]
        if isinstance(lanes, list):
            lanes = lanes[0]
        return pd.Series({
            "highway_type": str(highway) if highway else None,
            "lanes"       : float(lanes) if lanes else None,
        })
    except Exception:
        return pd.Series({"highway_type": None, "lanes": None})

df[["highway_type", "lanes"]] = df.apply(
    lambda row: nearest_street_features(row["lat"], row["lon"]), axis=1
)

print(f"\n✓ Done")
print(f"\nhighway_type fill : {df['highway_type'].notna().sum()}")
print(f"lanes fill        : {df['lanes'].notna().sum()}")
print(f"\nhighway_type distribution:")
print(df["highway_type"].value_counts())

✓ 1213 street segments loaded

✓ Done

highway_type fill : 2914
lanes fill        : 2623

highway_type distribution:
highway_type
secondary        1131
residential      1094
primary           516
unclassified       95
tertiary           62
living_street      13
tertiary_link       1
motorway_link       1
motorway            1
Name: count, dtype: int64


## 4.0 Save CSV

In [12]:
df.to_csv("csv files/TS-SD-street_width.csv", index=False, encoding="utf-8")

print(f"✓ {len(df)} records saved to 'csv files/TS-SD-street_width.csv'")

✓ 2914 records saved to 'csv files/TS-SD-street_width.csv'


In [11]:
df[["osm_id",  "label", "highway_type", "lanes"]].head(30)

,osm_id,label,highway_type,lanes
0,796338420,retail,secondary,1.0
1,258619240,retail,residential,2.0
2,2613887631,retail,secondary,1.0
3,3595084281,retail,secondary,1.0
4,1740694091,food_drink,secondary,3.0
5,9472159125,retail,secondary,1.0
6,2689589692,retail,secondary,1.0
7,368051619,leisure_entertainment,secondary,3.0
8,9674241492,retail,secondary,3.0
9,2689589690,food_drink,residential,1.0
